# Waveform Propagation Examples

This notebook demonstrates the new architecture for scalar field waveform propagation through galactic density profiles.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from utils.constants import SEC_TO_INEV
from utils.data_utils import read_medium_data, interpolate_data
from inputs.spectrum_sources import (CSVSpectrum, AnalyticSpectrum, SpectrumSource,
                                      TimeVaryingSpectrum, plot_spectrum)
from inputs.configs import PhysicsConfig, PropagationConfig
from utils.logging_utils import setup_logging, get_logger
from waveform.collection import WaveformCollection
from waveform.propagation import plot_spectrogram

# Enable inline plotting
%matplotlib inline

# Configure logging
setup_logging(log_file='propagation.log', level='INFO')
logger = get_logger()

## Example 1: Single Static Gaussian

Demonstrates a single Gaussian wavepacket propagating through the galactic density profile.

In [ ]:
from utils.constants import SOLAR_TO_EV

mass = 1e-19
burst_duration = 2 * np.pi * 100 / (mass * SEC_TO_INEV)

# Create spectrum source
Etot = SOLAR_TO_EV
delta_p = 20 * mass
amplitude = Etot / (np.sqrt(2*np.pi) * delta_p)
spectrum = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=2e2 * mass,
    width=delta_p,
    amplitude=amplitude
)

# Optionally plot the initial spectrum before propagation
plot_spectrum(spectrum, filename='gaussian_initial.pdf')
print("✓ Initial spectrum saved: gaussian_initial.pdf")

# Configure physics and propagation
physics = PhysicsConfig(mass=mass, coupling=1e20, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=2000)

# Create collection and propagate
N_points_spectrogram = 2000
collection = WaveformCollection(spectrum, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram)

print("✓ Waveform saved: waveform_plot.pdf")

# Plot spectrogram
avg_density, arrival_window = plot_spectrogram(N_points_spectrogram, results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], filename='spectrogram_single_gaussian.pdf')
print("✓ Spectrogram saved: spectrogram_single_gaussian.pdf")

logger.info("Single Gaussian propagation complete. Spectrogram saved.")

## Example 2: Time-Varying Gaussian

Demonstrates a Gaussian wavepacket with time-varying amplitude modulation.

In [ ]:
mass = 1e-19
burst_duration = 2 * np.pi * 100 / (mass * SEC_TO_INEV)

# Create base Gaussian spectrum
Etot = SOLAR_TO_EV
delta_p = 20 * mass
amplitude = Etot / (np.sqrt(2*np.pi) * delta_p)
base_gaussian = AnalyticSpectrum(
    'gaussian',
    mass=mass,
    num_points=1000,
    peak_momentum=2e2 * mass,
    width=2e1 * mass,
    amplitude=amplitude
)

# Create time-varying wrapper with Gaussian amplitude profile
N_time_steps = 20
amplitude_profile = [np.exp(-(i - N_time_steps/2)**2 / (N_time_steps/3)**2)
                     for i in range(N_time_steps)]

# Define time windows for each step
time_windows = [(i * burst_duration * SEC_TO_INEV / N_time_steps, (i + 1) * burst_duration * SEC_TO_INEV / N_time_steps)
                for i in range(N_time_steps)]

time_varying = TimeVaryingSpectrum(base_gaussian, amplitude_profile, time_windows)

# Configure and propagate
physics = PhysicsConfig(mass=mass, coupling=1e20, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=2000)

# Create collection and propagate
N_points_spectrogram = 2000
collection = WaveformCollection(time_varying, physics, propagation_config)
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram, save_waveform=False)

# Plot spectrogram
avg_density, arrival_window = plot_spectrogram(N_points_spectrogram, results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], filename='spectrogram_time_varying.pdf')
print("✓ Spectrogram saved: spectrogram_time_varying.pdf")

logger.info(f"Time-varying propagation complete with {N_time_steps} steps")

## Example 3: CSV Spectrum (Bosenova)

Demonstrates loading a spectrum from a CSV file (boson star collapse spectrum).

In [ ]:
from utils.constants import SEC_TO_INEV, KPC_TO_INEV, GCM3_TO_EV4

mass = 1e-19
burst_duration = 400 / (mass * SEC_TO_INEV)

# Load bosenova spectrum from CSV with scaling applied directly
spectrum = CSVSpectrum(
    'Spectra/BosonStarSpectrumRelOnly.txt',
    i_p=0, 
    i_A=1, 
    skip_header=False, 
    num_points=1000,
    scaled_momentum=mass,  # Scale dimensionless momentum by mass
    scaled_amplitude=lambda A: A * (1/1e-85)  # Normalization scaling
)

physics = PhysicsConfig(mass=mass, coupling=1e22, K=1e-3, burst_duration=burst_duration)
propagation_config = PropagationConfig('Galactic_Density_Profile.csv', density_num_points=1000)

# For bosenova, scale the density profile distance from 10kpc to 1pc (x/10000)
collection = WaveformCollection(spectrum, physics, propagation_config)

# Override the density profile to apply bosenova scaling
x, rho = read_medium_data('Galactic_Density_Profile.csv', i_R=0, i_rho=2)
x_interp, rho_interp = interpolate_data(x/10000, rho, 1000)  # Bosenova: Convert 10 kpc to 1 pc
collection.density_profile = [x_interp * KPC_TO_INEV, rho_interp * GCM3_TO_EV4]

N_points_spectrogram = 1000
results = collection.propagate_all(N_points_spectrogram=N_points_spectrogram, save_waveform=False)

# Plot spectrogram
avg_density, arrival_window = plot_spectrogram(N_points_spectrogram, results['t_min'], results['t_max'],
                 results['E'], results['spectrogram'], filename='spectrogram_bosenova.pdf')
print("✓ Spectrogram saved: spectrogram_bosenova.pdf")

logger.info("CSV (Bosenova) spectrum propagation complete. Spectrogram saved.")

## Summary

This notebook demonstrates the main use cases of the waveform propagation architecture:

1. **Single static spectrum**: Basic Gaussian wavepacket
2. **Time-varying spectrum**: Amplitude modulation over time
3. **CSV spectrum**: Loading external data (bosenova)

Each example produces:
- Initial spectrum plot (optional)
- Waveform plot showing φ(t) at Earth
- Spectrogram showing frequency vs. time evolution